In [1]:
import torch
import torch.nn as nn
import torchvision.transforms.functional as func

from torchvision.models import resnet18

class ResNet18Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = resnet18(weights=None)
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu)
        self.pool = resnet.maxpool
        self.encoder1 = resnet.layer1
        self.encoder2 = resnet.layer2
        self.encoder3 = resnet.layer3
        self.encoder4 = resnet.layer4
    def forward(self, x):
        x0 = self.stem(x)
        x1 = self.encoder1(self.pool(x0))
        x2 = self.encoder2(x1)
        x3 = self.encoder3(x2)
        x4 = self.encoder4(x3)
        return x0, x1, x2, x3, x4


class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        self.conv = nn.Sequential(
            nn.Conv2d(skip_channels + out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x, skip):
        x = self.up(x)
        skip = func.center_crop(skip, [x.shape[2], x.shape[3]])
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)

class ResNetUNet(nn.Module):
    def __init__(self, n_classes=1):
        super().__init__()
        self.encoder = ResNet18Encoder()

        self.decoder4 = DecoderBlock(512, 256, 256)
        self.decoder3 = DecoderBlock(256, 128, 128)
        self.decoder2 = DecoderBlock(128, 64, 64)
        self.decoder1 = DecoderBlock(64, 64, 32)

        self.final_conv = nn.Conv2d(32, n_classes, kernel_size=1)

    def forward(self, x):
        x0, x1, x2, x3, x4 = self.encoder(x)

        d4 = self.decoder4(x4, x3)
        d3 = self.decoder3(d4, x2)
        d2 = self.decoder2(d3, x1)
        d1 = self.decoder1(d2, x0)

        out = self.final_conv(d1)
        return out

In [2]:
### Fourier Domain Adaptation

@torch.no_grad()
def fda(src_img: torch.Tensor, tgt_img: torch.Tensor, beta) -> torch.Tensor:
    C, H, W = src_img.shape

    src_fft = torch.fft.fft2(src_img, dim=(-2, -1))
    tgt_fft = torch.fft.fft2(tgt_img, dim=(-2, -1))

    src_amp, src_phase = torch.abs(src_fft), torch.angle(src_fft)
    tgt_amp = torch.abs(tgt_fft)

    src_amp_shift = torch.fft.fftshift(src_amp, dim=(-2, -1))
    tgt_amp_shift = torch.fft.fftshift(tgt_amp, dim=(-2, -1))

    b_h = int(beta * H)
    b_w = int(beta * W)
    c_h, c_w = H // 2, W // 2
    h1, h2 = c_h - b_h, c_h + b_h
    w1, w2 = c_w - b_w, c_w + b_w

    src_amp_shift[..., h1:h2, w1:w2] = tgt_amp_shift[..., h1:h2, w1:w2]

    src_amp_new = torch.fft.ifftshift(src_amp_shift, dim=(-2, -1))
    src_fft_new = src_amp_new * torch.exp(1j * src_phase)
    src_in_tgt = torch.fft.ifft2(src_fft_new, dim=(-2, -1)).real

    return torch.clamp(src_in_tgt, 0.0, 255.0)

In [3]:
from torch.utils.data import Dataset
import tifffile as tif
import random
from torchvision import transforms
import os

random.seed(42)

normalize_05 = transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5))
to_tensor = transforms.ToTensor()
imageTransform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class LandslideDataset(Dataset):
    def __init__(self, img_list, mask_list):
        self.img_list = img_list
        self.mask_list = mask_list
    
    def __len__(self):
        return len(self.img_list)
    
    def __getitem__(self, index):
        img_dir = self.img_list[index]
        mask_dir = self.mask_list[index]

        img = tif.imread(img_dir)
        mask = tif.imread(mask_dir)

        img = imageTransform(img)
        mask = (to_tensor(mask) > 0).float()

        return img, mask
    
class SourceDataset(Dataset):
    def __init__(self, source_ds, target_ds, beta):
        self.source = source_ds
        self.target = target_ds
        self.beta = beta
    
    def __len__(self):
        return len(self.source)
    
    def sample_target_img(self):
        random.seed()
        tgt_img = random.sample(self.target, 1)

        img = tif.imread(tgt_img)
        img = to_tensor(img)
        return img
    
    def __getitem__(self, idx):
        src = self.source[idx]
        mask = src.replace("img", "mask")

        src_img = to_tensor(tif.imread(src)).float().to(device, non_blocking=True)
        src_mask = (to_tensor(tif.imread(mask)) > 0).float().to(device, non_blocking=True)

        tgt_img = self.sample_target_img().float().to(device, non_blocking=True)

        src_img = (fda(src_img*255, tgt_img*255, beta=self.beta)/255.0).clamp(0,1)
        src_img = normalize_05(src_img)

        return src_img, src_mask

path = "../../data/"
limit = 100
regions = [f for f in os.listdir(path) if (os.path.isdir(os.path.join(path, f)) and f != "study areas shp")]

regions_dict = dict()

for region in regions:
    dataset_dir = path + region
    image_dir = os.path.join(dataset_dir, "img")
    img_list = os.listdir(image_dir)

    all_images = sorted(os.path.join(image_dir, f) for f in img_list)

    if len(all_images) > limit:
        all_images = random.sample(all_images, limit)

    regions_dict[region] = all_images

In [4]:
def getMetrics(TP, TN, FP, FN):
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
        
    iou_1  = TP / (TP + FP + FN + 1e-8)
    iou_0 = TN / (TN + FP + FN + 1e-8)
    miou = (iou_0 + iou_1) / 2 
        
    oa = (TP + TN)/(TP + TN + FP + FN + 1e-8)
    
    return precision, recall, f1, iou_1, miou, oa

def dice_loss(pred, target, smooth=1.):
    pred = torch.sigmoid(pred)
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    intersection = (pred_flat * target_flat).sum()
    return 1 - ((2. * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth))


### COMPUTE R (NEGATIVE PIXELS / POSITIVE PIXELS)
r = 4.5
pos_weight = torch.tensor([r]).to(device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [5]:
import torch.nn.functional as F

def train_model(model, optimizer, train_loader, val_loader, epochs, model_path, region_name, threshold=0.6):
    best_iou = 0.0
    patience = 10
    counter = 0

    print(f'region, epoch, train_loss, val_loss, precision, recall, f1, iou, miou, oa')

    for epoch in range(epochs):
        model.train()
        running_loss, n_samples = 0.0, 0

        for images, masks in train_loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True).float()

            optimizer.zero_grad(set_to_none=True)

            logits = model(images)
            if logits.shape[-2:] != masks.shape[-2:]:
                logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

            bce  = criterion(logits, masks)
            dice = dice_loss(logits, masks)
            loss = bce + dice

            loss.backward()
            optimizer.step()

            running_loss += loss.item() * masks.numel()
            n_samples += masks.numel()
        
        training_loss = running_loss / max(1, n_samples)

        model.eval()
        val_loss, val_num = 0.0, 0
        
        TP, FP, FN, TN = 0,0,0,0

        with torch.no_grad():
            for images, masks in val_loader:
                images = images.to(device, non_blocking=True)
                masks = masks.to(device, non_blocking=True).float()

                logits = model(images)
                if logits.shape[-2:] != masks.shape[-2:]:
                    logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

                bce  = criterion(logits, masks)
                dice = dice_loss(logits, masks)
                loss = bce + dice
                val_loss += loss.item()

                preds = torch.sigmoid(logits) > threshold

                preds_flat = preds.view(-1)
                mask_flat = masks.view(-1)

                TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
                FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
                FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
                TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()
                
                val_num += 1
             
        precision, recall, f1, iou, miou, oa = getMetrics(TP, TN, FP, FN)

        print(f'{region_name}, {epoch+1}, {training_loss :.3f}, {val_loss / val_num :.3f}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}')

        if iou > best_iou:
            best_iou = iou
            counter = 0
            
            torch.save(model.state_dict(), model_path)
        elif iou != 0.0:
            counter += 1
            if counter >= patience:
                break

def test_model(model, test_loader, threshold=0.6):
    TP, FP, FN, TN = 0,0,0,0

    model.eval()

    with torch.no_grad():
        for images, masks in test_loader:
            images = images.to(device, non_blocking=True)
            masks = masks.to(device, non_blocking=True).float()

            logits = model(images)
            if logits.shape[-2:] != masks.shape[-2:]:
                logits = F.interpolate(logits, size=masks.shape[-2:], mode="bilinear", align_corners=False)

            preds = torch.sigmoid(logits) > threshold
                    
            preds_flat = preds.view(-1)
            mask_flat = masks.view(-1)

            TP += ((preds_flat == 1) & (mask_flat == 1)).sum().item()
            FP += ((preds_flat == 1) & (mask_flat == 0)).sum().item()
            FN += ((preds_flat == 0) & (mask_flat == 1)).sum().item()
            TN += ((preds_flat == 0) & (mask_flat == 0)).sum().item()

    return getMetrics(TP, TN, FP, FN)

In [6]:
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import torch.optim as optim

batch_size = 16

beta = 0.01
beta_name = "01"

output = "source_region,target_region,precision,recall,f1,iou,miou,oa"

for source_region in regions_dict:
    source_images = regions_dict[source_region]
    source_masks = [f.replace("img", "mask") for f in source_images]

    train_img, val_img, train_mask, val_mask = train_test_split(
        source_images, source_masks, test_size=.2, random_state=42
    )

    for target_region in regions_dict:
        if source_region != target_region:
            epochs = 40

            model = ResNetUNet().to(device)

            target_test_img = regions_dict[target_region]
            target_test_mask = [f.replace("img", "mask") for f in target_test_img]

            train_dataset = SourceDataset(train_img, target_test_img, beta)
            val_dataset = LandslideDataset(val_img, val_mask)
            test_dataset = LandslideDataset(target_test_img, target_test_mask)

            trainLoader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
            valLoader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
            testLoader = DataLoader(test_dataset, batch_size=batch_size,shuffle=False,num_workers=0)

            optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

            model_path = f"../../models/new_fda/{beta_name}/{source_region}_{target_region}.pth"

            train_model(model, optimizer, trainLoader, valLoader, epochs, model_path, source_region)

            model.load_state_dict(torch.load(model_path, map_location=device))

            precision, recall, f1, iou, miou, oa = test_model(model, testLoader)
            result = f'{source_region}, {target_region}, {precision:.4f}, {recall:.4f}, {f1:.4f}, {iou:.4f}, {miou:.4f}, {oa:.4f}'
            print(result)
            output += f'\n{result}'

with open(f"../../results/new_fda/{beta_name}.csv", "w") as f:
    f.write(output)


region, epoch, train_loss, val_loss, precision, recall, f1, iou, miou, oa
Hokkaido Iburi-Tobu, 1, 1.783, 1.768, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 2, 1.752, 1.760, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 3, 1.725, 1.739, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 4, 1.683, 1.661, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 5, 1.636, 1.509, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 6, 1.583, 2.960, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 7, 1.527, 4.241, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 8, 1.495, 2.393, 0.0000, 0.0000, 0.0000, 0.0000, 0.4686, 0.9372
Hokkaido Iburi-Tobu, 9, 1.432, 1.933, 0.0873, 0.0045, 0.0085, 0.0043, 0.4694, 0.9345
Hokkaido Iburi-Tobu, 10, 1.352, 1.912, 0.3361, 0.0214, 0.0402, 0.0205, 0.4782, 0.9359
Hokkaido Iburi-Tobu, 11, 1.405, 1.171, 0.6588, 0.3006, 0.4129, 0.2601, 0.60